In [1]:
import pandas as pd
import json
import os
import time
from pathlib import Path
from typing import Any
from pathlib import Path
from openai import OpenAI

In [2]:
df = pd.read_csv("grocery_negative.csv")
pd.set_option("display.max_colwidth", 200)
df.head(3)

,address,name_ru,rating,rubrics,text,is_grocery,id,len
0,"Краснодарский край, Сочи, жилой район Адлер, улица Молокова, 30, корп. 2",Черемушки,2,Супермаркет;Магазин продуктов,Самый неопрятный магазин.Еще и тараканы бегают,True,74,46
1,"Свердловская область, Екатеринбург, улица Черепанова, 12А",Магнит косметик,2,Магазин продуктов;Магазин хозтоваров и бытовой химии;Супермаркет,"Вместо мегамарта открыли очередной Магнит. Всё,как везде. Вечно очередь стоит,так как работает 1 кассир,остальные полки загружают. Понятно,дефицит кадров. Столкнулась с привычной историей- несоотв...",True,935,508
2,"Московская область, Раменский городской округ, деревня Петровское, Школьная улица, 1Г",Дикси,1,Супермаркет;Магазин продуктов,"Отвратительный магазин, кассы не удобные и очень маленькие + тесные, касиры хамовитые, мясяца 3 у них небыло ни корзин не тележек.",True,960,130


In [3]:
df.drop(columns=["address", "rating", "rubrics", "is_grocery"], inplace=True)
df.head(3)

,name_ru,text,id,len
0,Черемушки,Самый неопрятный магазин.Еще и тараканы бегают,74,46
1,Магнит косметик,"Вместо мегамарта открыли очередной Магнит. Всё,как везде. Вечно очередь стоит,так как работает 1 кассир,остальные полки загружают. Понятно,дефицит кадров. Столкнулась с привычной историей- несоотв...",935,508
2,Дикси,"Отвратительный магазин, кассы не удобные и очень маленькие + тесные, касиры хамовитые, мясяца 3 у них небыло ни корзин не тележек.",960,130


In [4]:
def load_env_file() -> None:
    for env_path in [Path(".env"), Path("x5_reviews/.env")]:
        if not env_path.exists():
            continue

        for line in env_path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            os.environ.setdefault(key, value)

        break


load_env_file()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
OPENAI_BASE_URL = (os.getenv("OPENAI_BASE_URL") or "").strip() or None
OPENAI_MODEL = (os.getenv("OPENAI_MODEL", "gpt-4.1-mini") or "").strip()
BATCH_SIZE = 10

LABEL_COLUMNS = [
    "product_quality",
    "staff_service",
    "queues_checkout",
    "assortment_availability",
    "store_cleanliness",
    "other_problem",
]

SYSTEM_PROMPT = f"""
Ты размечаешь отзывы о продуктовых магазинах в multi-label формате.
Один отзыв может относиться сразу к нескольким классам.
Верни только JSON-массив без markdown и пояснений.

Для каждого объекта верни поля:
- id: исходный id отзыва
- product_quality: true/false
- staff_service: true/false
- queues_checkout: true/false
- assortment_availability: true/false
- store_cleanliness: true/false
- other_problem: true/false

Правила:
- Ставь true только если проблема явно есть в тексте.
- Если проблема не упомянута, ставь false.
- other_problem=true ставь только если есть негативная проблема, не попадающая в остальные классы.
- Если текст нейтральный, пустой или недостаточно конкретный, все метки должны быть false.

Разметь следующие отзывы и верни JSON-массив той же длины, где каждому input id соответствует один output объект:\n\n"
""".strip()

print(OPENAI_BASE_URL, OPENAI_MODEL)

https://openrouter.ai/api/v1 google/gemma-4-26b-a4b-it


In [5]:
def validate_item(item, expected_id):
    if item.get("id") != expected_id:
        raise ValueError(f"Expected id={expected_id}, got id={item.get('id')}")

    result = {"id": expected_id}
    for label in LABEL_COLUMNS:
        value = item.get(label)
        if not isinstance(value, bool):
            raise ValueError(f"{label} must be bool for id={expected_id}")
        result[label] = value

    result["label_status"] = "True"
    result["label_error"] = None
    return result


def build_batch_prompt(batch_df):
    records = []
    for row in batch_df[["id", "name_ru", "text"]].itertuples(index=False):
        records.append(
            {
                "id": row.id,
                "text": row.text,
            }
        )

    return json.dumps(records, ensure_ascii=False, indent=2)

In [6]:
import asyncio

ASYNC_CONCURRENCY = 5

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)


def extract_json_array(raw_text):
    raw_text = (raw_text or "").strip()

    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        raw_text = raw_text.replace("json\n", "", 1).strip()

    start = raw_text.find("[")
    end = raw_text.rfind("]")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"JSON array not found in response: {raw_text[:400]}")

    parsed_list = json.loads(raw_text[start : end + 1])
    if not isinstance(parsed_list, list):
        raise ValueError("Response is not a list")
    return parsed_list


def label_batch(batch_df: pd.DataFrame) -> dict["status":str, "rows":any]:
    try:
        # prepare
        expected_ids = batch_df["id"].tolist()
        user_prompt = build_batch_prompt(batch_df)
        # send to llm
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        # parse and check
        print(response.usage)
        raw = response.choices[0].message.content or ""
        parsed_list = extract_json_array(raw)
        items_by_id = {}

        for item in parsed_list:
            item_id = item.get("id")
            items_by_id[item_id] = item

        if items_by_id.keys() != set(expected_ids):
            raise ValueError(
                f"Expected ids {expected_ids}, got {list(items_by_id.keys())}"
            )

        validated_rows = []
        for expected_id in expected_ids:
            validated_rows.append(validate_item(items_by_id[expected_id], expected_id))

        return {"status": "success", "rows": validated_rows}
    except Exception as error:
        error_message = str(error)[:100]
        print(f"Error processing batch ids {expected_ids}: {error_message[:200]}")
    return {"status": "error", "rows": []}


async def label_batch_async(
    batch_df: pd.DataFrame, semaphore: asyncio.Semaphore
) -> dict[str, Any]:
    async with semaphore:
        print(f"Sending batch: {batch_df['id'].tolist()[0]}")
        res = await asyncio.to_thread(label_batch, batch_df)
        if res.get("status") != "success":
            print(f"Batch labeling failed for batch id {batch_df['id'].tolist()[0]}")
        return res


async def label_dataframe_async(df, dataframes, concurrency):
    semaphore = asyncio.Semaphore(concurrency)
    batches = dataframes
    tasks = []
    for batch_df in batches:
        tasks.append(label_batch_async(batch_df, semaphore))
    results = await asyncio.gather(*tasks)
    all_rows = []
    bad_count = 0
    for batch in results:
        if batch.get("status") == "success":
            for row in batch.get("rows", []):
                all_rows.append(row)
        else:
            bad_count += 1
    print(f"Erorr at {bad_count} batches")

    labels_result_df = pd.DataFrame(all_rows)
    return labels_result_df

In [7]:
"""
df_temp = df.copy().head(32)
print(len(df_temp))
batches = []
for start in range(0, len(df_temp), BATCH_SIZE):
    batch_df = df_temp.iloc[start : start + BATCH_SIZE].copy()
    if not batch_df.empty:
        batches.append(batch_df)

result_df = await label_dataframe_async(df_temp, batches, ASYNC_CONCURRENCY)
"""

'\ndf_temp = df.copy().head(32)\nprint(len(df_temp))\nbatches = []\nfor start in range(0, len(df_temp), BATCH_SIZE):\n    batch_df = df_temp.iloc[start : start + BATCH_SIZE].copy()\n    if not batch_df.empty:\n        batches.append(batch_df)\n\nresult_df = await label_dataframe_async(df_temp, batches, ASYNC_CONCURRENCY)\n'

In [8]:
result_df.head(50)

NameError: name 'result_df' is not defined

In [9]:
load_env_file()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
OPENAI_BASE_URL = (os.getenv("OPENAI_BASE_URL") or "").strip() or None
OPENAI_MODEL = (os.getenv("OPENAI_MODEL", "gpt-4.1-mini") or "").strip()
BATCH_SIZE = 20
print(OPENAI_MODEL, OPENAI_BASE_URL)
input()

df_temp = df.copy()
print(len(df_temp))
batches = []
for start in range(0, len(df_temp), BATCH_SIZE):
    batch_df = df_temp.iloc[start : start + BATCH_SIZE].copy()
    if not batch_df.empty:
        batches.append(batch_df)

result_df = await label_dataframe_async(df_temp, batches, ASYNC_CONCURRENCY)

google/gemma-4-26b-a4b-it https://openrouter.ai/api/v1
2196
Sending batch: 74
Sending batch: 7113
Sending batch: 10455
Sending batch: 15218
Sending batch: 19117
CompletionUsage(completion_tokens=1480, prompt_tokens=2675, total_tokens=4155, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=0.0009303525, is_byok=False, cost_details={'upstream_inference_cost': 0.00093975, 'upstream_inference_prompt_cost': 0.00034775, 'upstream_inference_completions_cost': 0.000592})
Sending batch: 26078
CompletionUsage(completion_tokens=1504, prompt_tokens=2900, total_tokens=4404, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=Pro

In [11]:
print(len(df))
print(len(result_df))

2196
2136


In [12]:
result_df.to_csv("first_try.csv")

In [14]:
labled = result_df["id"].to_list()
all = df["id"].to_list()
to_label = [i for i in all if i not in labled]
to_label

[127402,
 127435,
 127811,
 127888,
 128070,
 128184,
 128341,
 128343,
 128352,
 128811,
 128959,
 129066,
 129455,
 130530,
 130695,
 131057,
 131280,
 131610,
 131718,
 131911,
 133056,
 133201,
 133400,
 133722,
 134206,
 134879,
 134947,
 135029,
 135153,
 135154,
 135703,
 136057,
 136132,
 136396,
 136445,
 136512,
 136698,
 136701,
 137046,
 137065,
 137430,
 137546,
 137558,
 138195,
 138410,
 138451,
 138561,
 138661,
 138805,
 138809,
 138811,
 138813,
 139084,
 139262,
 139692,
 140028,
 140162,
 140249,
 140380,
 140382]

In [17]:
df_to_label = df[df["id"].isin(to_label)].copy()
len(df_to_label)

60

In [18]:
load_env_file()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
OPENAI_BASE_URL = (os.getenv("OPENAI_BASE_URL") or "").strip() or None
OPENAI_MODEL = (os.getenv("OPENAI_MODEL", "gpt-4.1-mini") or "").strip()
BATCH_SIZE = 20
print(OPENAI_MODEL, OPENAI_BASE_URL)
input()

df_temp = df_to_label
print(len(df_temp))
batches = []
for start in range(0, len(df_temp), BATCH_SIZE):
    batch_df = df_temp.iloc[start : start + BATCH_SIZE].copy()
    if not batch_df.empty:
        batches.append(batch_df)

result_df1 = await label_dataframe_async(df_temp, batches, ASYNC_CONCURRENCY)

google/gemma-4-26b-a4b-it https://openrouter.ai/api/v1
60
Sending batch: 127402
Sending batch: 133056
Sending batch: 137430
CompletionUsage(completion_tokens=1524, prompt_tokens=2974, total_tokens=4498, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=0.00123282225, is_byok=False, cost_details={'upstream_inference_cost': 0.001245275, 'upstream_inference_prompt_cost': 0.000483275, 'upstream_inference_completions_cost': 0.000762})
CompletionUsage(completion_tokens=1524, prompt_tokens=2570, total_tokens=4094, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache

In [19]:
len(result_df1)

60

In [20]:
result_df1.to_csv("second_try.csv")

In [42]:
df1 = pd.read_csv("first_try.csv")
df2 = pd.read_csv("second_try.csv")
merged = pd.concat([df1, df2])
print(len(merged))

2196


In [43]:
merged.head(10)

,Unnamed: 0,id,product_quality,staff_service,queues_checkout,assortment_availability,store_cleanliness,other_problem,label_status,label_error
0,0,74,False,False,False,False,True,False,True,NaN
1,1,935,False,False,True,False,False,True,True,NaN
2,2,960,False,True,True,False,False,False,True,NaN
3,3,1178,False,True,False,False,False,True,True,NaN
4,4,1263,False,False,False,False,False,True,True,NaN
5,5,2012,True,False,False,False,False,False,True,NaN
6,6,2077,False,False,False,False,False,True,True,NaN
7,7,2202,False,True,False,False,False,False,True,NaN
8,8,2280,False,True,False,False,False,False,True,NaN
9,9,2532,True,False,False,False,False,False,True,NaN


In [44]:
merged["label_status"].value_counts()

label_status
True    2196
Name: count, dtype: int64

In [45]:
merged.drop(columns=["Unnamed: 0", "label_status", "label_error"], inplace=True)
merged.head(3)

,id,product_quality,staff_service,queues_checkout,assortment_availability,store_cleanliness,other_problem
0,74,False,False,False,False,True,False
1,935,False,False,True,False,False,True
2,960,False,True,True,False,False,False


In [46]:
df.head(10)

,name_ru,text,len
id,,,
74,Черемушки,Самый неопрятный магазин.Еще и тараканы бегают,46
935,Магнит косметик,"Вместо мегамарта открыли очередной Магнит. Всё,как везде. Вечно очередь стоит,так как работает 1 кассир,остальные полки загружают. Понятно,дефицит кадров. Столкнулась с привычной историей- несоотв...",508
960,Дикси,"Отвратительный магазин, кассы не удобные и очень маленькие + тесные, касиры хамовитые, мясяца 3 у них небыло ни корзин не тележек.",130
1178,Красное&Белое,"Добрый вечер! Зачем ваши сотрудники втюхивают, что не нужно. Пришёл в ваш магазин, взял что мне нужно. Взял водку Тундра 309,89 руб. Они мне настоятельно порекомендовали другую водку Сенега или Пе...",731
1263,Дикси,"Постоянно ценники не соответствуют цене на кассе. Часто надо смотреть цену на приборе, т.к.нет ценника,а это не удобно.",119
2012,Бисквит,Купили «Красный бархат» 29 апреля в 17:00 на улице Нахимова. Торт не соответствует своим заявленным качествам. Не сладкий. Коржи странные и странного цвета. Крем точно не натуральный сливочный кре...,310
2077,Дары моря,"Их постоянно обкрадывают на икру, вешаешь 100 грам, её сразу уносят на кассу, если орут чья то как то стыдно, когда молча уносят можно на кассе ее забыть если много чего другого купил, или тебе ка...",268
2202,Ладоград,"В этот магазин ходила за продуктами с самого его открытия, живу в пешей доступности. Всем его рекомендовала.\nНо в последнее время частенько не могу туда попасть. Закрываются за 15- 20 минут до 2...",812
2280,Микей,"Качество продуктов и чистота в магазине радуют, но вот эти временные продавцы, которые приходят в магазин на 1-2 дня это ужас. Набирают хрен пойми откуда. Недавно покупала прокладки и мне довольно...",595


In [47]:
df3 = pd.read_csv("grocery_negative.csv")
print(df3.head(3))
print(merged.head(3))

                                                                                 address  \
0               Краснодарский край, Сочи, жилой район Адлер, улица Молокова, 30, корп. 2   
1                              Свердловская область, Екатеринбург, улица Черепанова, 12А   
2  Московская область, Раменский городской округ, деревня Петровское, Школьная улица, 1Г   

           name_ru  rating  \
0        Черемушки       2   
1  Магнит косметик       2   
2            Дикси       1   

                                                            rubrics  \
0                                     Супермаркет;Магазин продуктов   
1  Магазин продуктов;Магазин хозтоваров и бытовой химии;Супермаркет   
2                                     Супермаркет;Магазин продуктов   

                                                                                                                                                                                                      text  \
0                  

In [48]:
final_df = df3.merge(merged, on="id")
len(final_df)

2196

In [49]:
final_df.head(5)

,address,name_ru,rating,rubrics,text,is_grocery,id,len,product_quality,staff_service,queues_checkout,assortment_availability,store_cleanliness,other_problem
0,"Краснодарский край, Сочи, жилой район Адлер, улица Молокова, 30, корп. 2",Черемушки,2,Супермаркет;Магазин продуктов,Самый неопрятный магазин.Еще и тараканы бегают,True,74,46,False,False,False,False,True,False
1,"Свердловская область, Екатеринбург, улица Черепанова, 12А",Магнит косметик,2,Магазин продуктов;Магазин хозтоваров и бытовой химии;Супермаркет,"Вместо мегамарта открыли очередной Магнит. Всё,как везде. Вечно очередь стоит,так как работает 1 кассир,остальные полки загружают. Понятно,дефицит кадров. Столкнулась с привычной историей- несоотв...",True,935,508,False,False,True,False,False,True
2,"Московская область, Раменский городской округ, деревня Петровское, Школьная улица, 1Г",Дикси,1,Супермаркет;Магазин продуктов,"Отвратительный магазин, кассы не удобные и очень маленькие + тесные, касиры хамовитые, мясяца 3 у них небыло ни корзин не тележек.",True,960,130,False,True,True,False,False,False
3,"Свердловская область, Новоуральск, улица Ленина, 62",Красное&Белое,2,Магазин алкогольных напитков;Магазин продуктов,"Добрый вечер! Зачем ваши сотрудники втюхивают, что не нужно. Пришёл в ваш магазин, взял что мне нужно. Взял водку Тундра 309,89 руб. Они мне настоятельно порекомендовали другую водку Сенега или Пе...",True,1178,731,False,True,False,False,False,True
4,"Москва, Скобелевская улица, 23",Дикси,1,Супермаркет;Магазин продуктов,"Постоянно ценники не соответствуют цене на кассе. Часто надо смотреть цену на приборе, т.к.нет ценника,а это не удобно.",True,1263,119,False,False,False,False,False,True


In [50]:
final_df.to_csv("labled.csv")